# P3 single-date training: Prithvi + U-Net + WCE

Notebook này fine-tune nhánh **T=1** trên ảnh Xuân Thủy ngày 21/09/2026.
Encoder khởi tạo từ Prithvi-EO-2.0-300M-TL chính thức; U-Net decoder và head
khởi tạo mới. Notebook không lấy trọng số P2 T=6, không tạo map và không dùng
điểm kiểm chứng để chọn checkpoint.

P3 giữ nguyên split, guard, patch schedule, WCE, seed, optimizer và tiêu chí
validation macro-F1 của P2. Dữ liệu sáu ngày vẫn được audit để tái sử dụng đúng
common-validity/split evidence; tensor đưa vào model chỉ chứa ngày 21/09.

## 1. Mount Drive và checkout mã nguồn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/hongphuc-maker/xuan-thuy-prithvi-segmentation.git'
REPO_REF = 'main'  # Replace with the approved commit SHA before the scientific run.
REPO_ROOT = Path('/content/xuan-thuy-prithvi-segmentation')

if not (REPO_ROOT / '.git').exists():
    subprocess.check_call(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(REPO_ROOT)])
else:
    dirty = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('The Colab repository copy has local changes; use a fresh runtime or review them.')

subprocess.check_call(['git', '-C', str(REPO_ROOT), 'fetch', '--depth=1', 'origin', REPO_REF])
subprocess.check_call(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', 'FETCH_HEAD'])

REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', REPO_COMMIT)


## 2. Cài môi trường

Lần đầu cell này hoàn tất cài đặt, runtime sẽ tự restart để tránh trộn hai phiên
bản NumPy trong RAM. Sau restart, chạy lại notebook từ đầu; marker trong
`/content` khiến cell không cài lại lần nữa.

In [ ]:
import importlib.metadata
import os
import signal

ENV_MARKER = Path('/content/.prithvi_p3_single_env_v1')
if not ENV_MARKER.exists():
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'numpy==2.3.5', 'terratorch==1.2.12', 'torchgeo==0.9.0', 'timm==1.0.30',
        'segmentation-models-pytorch==0.5.0', 'huggingface-hub==1.33.0',
        'einops==0.8.2', 'rasterio==1.5.2', 'pandas==2.2.3',
        'matplotlib>=3.8,<4', 'PyYAML>=6,<7', 'geopandas>=1,<2', 'tqdm>=4.66,<5',
        'git+https://github.com/hongphuc-maker/xuan-thuy-dmi-segmentation.git@0775770ae87b83b7d2982ed727226128685f3fba'
    ])
    ENV_MARKER.write_text('installed; restart required', encoding='utf-8')
    print('Environment installed. Colab will restart now; then run all cells from the top again.')
    os.kill(os.getpid(), signal.SIGKILL)

P3_SOURCE = REPO_ROOT / 'reference' / 'wce_completed'
sys.path.insert(0, str(P3_SOURCE))
sys.path.insert(0, str(REPO_ROOT))
print({name: importlib.metadata.version(name) for name in [
    'numpy', 'torch', 'terratorch', 'torchgeo', 'timm', 'rasterio', 'pandas'
]})
print('P3 source:', P3_SOURCE)


## 3. Khóa config, dữ liệu và run directory riêng

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Prithvi_Colab')
DATA_ROOT = DRIVE_ROOT / 'data' / 'training_p2'
RUN_ID = 'P3_single_sep21_UNet_WCE_REPRO'
RUN_ROOT = DRIVE_ROOT / 'runs' / RUN_ID
CONFIG_PATH = REPO_ROOT / 'configs' / 'experiments' / 'p3_prithvi_unet_wce_single.json'

assert CONFIG_PATH.is_file(), CONFIG_PATH
assert DATA_ROOT.is_dir(), (
    f'{DATA_ROOT} does not exist. Point DATA_ROOT to the folder containing the '
    'six locked training GeoTIFFs and rasterized_vector_categorized.tif.'
)
print('DATA_ROOT:', DATA_ROOT)
print('RUN_ROOT:', RUN_ROOT)
print('CONFIG_PATH:', CONFIG_PATH)


## 4. Audit dữ liệu và xác nhận T=1

In [ ]:
import json
from IPython.display import display
from prithvi_xtseg.training import open_experiment

experiment = open_experiment(CONFIG_PATH, DATA_ROOT, RUN_ROOT)
display(experiment.scene.audit)
assert experiment.config['active_dates'] == ['2026-09-21']
assert experiment.scene.audit['shared_input_pixels'] == 0
print('Model active dates:', experiment.config['active_dates'])
print('Train patches:', len(experiment.manifest))
print('Validation windows:', len(experiment.val))


## 5. GPU smoke test trước optimizer update

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU')
model = experiment.load_model(pretrained=not (RUN_ROOT / 'latest_resume.pt').exists())
smoke = experiment.smoke_test()
display(smoke)
assert smoke['input_shape'] == [1, 6, 1, 224, 224]
assert smoke['output_shape'] == [1, 11, 224, 224]


## 6. Train hoặc resume P3

Sau khi kiểm tra `RUN_ROOT`, audit và smoke test, đổi `RUN_TRAINING=True`.
`latest_resume.pt` chỉ resume chính run P3 này; không đọc checkpoint P2.

In [ ]:
RUN_TRAINING = False

if not RUN_TRAINING:
    print('Training is disabled. Review the audit, then set RUN_TRAINING=True.')
else:
    training_result = experiment.train()
    display(training_result)


## 7. Đọc model-selection artifacts, không tạo map

In [ ]:
for filename in [
    'TRAINING_PROGRESS.json', 'TRAINING_COMPLETE.json',
    'validation_history.csv', 'latest_validation_per_class.csv'
]:
    path = RUN_ROOT / filename
    if path.exists():
        print('\n', filename)
        if path.suffix == '.json':
            display(json.loads(path.read_text()))
        else:
            import pandas as pd
            display(pd.read_csv(path).tail(20))

for checkpoint in ['best_accepted.pt', 'best_any.pt', 'latest_resume.pt']:
    path = RUN_ROOT / checkpoint
    print(checkpoint, 'exists=' + str(path.exists()))
print('This notebook stops before map inference by design.')
